In [31]:
import pandas as pd
import numpy as np
import os

In [32]:
df = pd.read_csv("../data/processed/train_clean.csv", parse_dates=["Date"], dtype={'StateHoliday': str})

print('Shape:', df.shape)
print(f'Date range: {df['Date'].min()} to {df['Date'].max()}')
df.head()

Shape: (844338, 18)
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,0.0,0.0,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,0.0,0.0,NaN


In [33]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = df['DayOfWeek'].isin([6, 7]).astype(int)
df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
df['IsMonthEnd'] = df['Date'].dt.is_month_end.astype(int)

print("New columns added:")
print(df[["Date","Year","Month","Day","Week","IsWeekend","IsMonthStart","IsMonthEnd"]].head())

New columns added:
        Date  Year  Month  Day  Week  IsWeekend  IsMonthStart  IsMonthEnd
0 2015-07-31  2015      7   31    31          0             0           1
1 2015-07-31  2015      7   31    31          0             0           1
2 2015-07-31  2015      7   31    31          0             0           1
3 2015-07-31  2015      7   31    31          0             0           1
4 2015-07-31  2015      7   31    31          0             0           1


In [34]:
df['CompetitionOpen'] = (
    12 * (df['Year'] - df['CompetitionOpenSinceYear']) +
    (df['Month'] - df['CompetitionOpenSinceMonth'])
)

df['CompetitionOpen'] = df['CompetitionOpen'].apply(lambda x: max(x, 0))

print(df["CompetitionOpen"].describe().round(2))
print("\nSample:")
print(df[["Year","Month","CompetitionOpenSinceYear","CompetitionOpenSinceMonth","CompetitionOpen"]].head())


count    844338.00
mean       7731.47
std       11229.44
min           0.00
25%          29.00
50%          91.00
75%       24163.00
max       24187.00
Name: CompetitionOpen, dtype: float64

Sample:
   Year  Month  CompetitionOpenSinceYear  CompetitionOpenSinceMonth  \
0  2015      7                    2008.0                        9.0   
1  2015      7                    2007.0                       11.0   
2  2015      7                    2006.0                       12.0   
3  2015      7                    2009.0                        9.0   
4  2015      7                    2015.0                        4.0   

   CompetitionOpen  
0             82.0  
1             92.0  
2            103.0  
3             70.0  
4              3.0  


In [35]:
promo_month_map = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "May": 5, "Jun": 6, "Jul": 7, "Aug": 8,
    "Sept": 9, "Oct": 10, "Nov": 11, "Dec": 12
}

def is_promo2_active(row):
    """
    Determine whether Promo2 is active for a store on a given date.

    Returns:
        1 if:
        - The store participates in Promo2,
        - The current month is included in the store's PromoInterval,
        - The current date is on or after the Promo2 start date.

        0 otherwise.
    """
    
    if row['Promo2'] == 0:
        return 0
    if row['PromoInterval'] == 'None':
        return 0
    
    promo_months = [promo_month_map[m] for m in row['PromoInterval'].split(',')]

    if row['Month'] in promo_months:
        if row['Year'] > row['Promo2SinceYear']:
            return 1
        if row['Year'] == row['Promo2SinceYear'] and row['Week'] >= row['Promo2SinceWeek']:
            return 1
    return 0

df["Promo2Active"] = df.apply(is_promo2_active, axis=1)

print("Promo2Active value counts:")
print(df["Promo2Active"].value_counts())
print("\nActive rate:", round(df["Promo2Active"].mean() * 100, 2), "%")


Promo2Active value counts:
Promo2Active
0    718062
1    126276
Name: count, dtype: int64

Active rate: 14.96 %


In [36]:
df["StoreType"]    = df["StoreType"].map({"a": 0, "b": 1, "c": 2, "d": 3})
df["Assortment"]   = df["Assortment"].map({"a": 0, "b": 1, "c": 2})
df["StateHoliday"] = df["StateHoliday"].map({"0": 0, "a": 1, "b": 2, "c": 3})

print("Encoded value counts:")
print("StoreType:", df["StoreType"].value_counts().to_dict())
print("Assortment:", df["Assortment"].value_counts().to_dict())
print("StateHoliday:", df["StateHoliday"].value_counts().to_dict())

Encoded value counts:
StoreType: {0: 457042, 3: 258768, 2: 112968, 1: 15560}
Assortment: {0: 444875, 2: 391254, 1: 8209}
StateHoliday: {0: 843428, 1: 694, 2: 145, 3: 71}


In [ ]:
cols_to_drop = [
    "Date",
    "Open",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval"
]

df = df.drop(columns=cols_to_drop)

print("Final shape:", df.shape)
print("Columns:", df.columns.tolist())

Final shape: (844338, 20)
Columns: ['Store', 'DayOfWeek', 'Sales', 'Customers', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'Year', 'Month', 'Day', 'Week', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd', 'CompetitionOpen', 'Promo2Active']


Final shape: (844338, 20)
Columns: ['Store', 'DayOfWeek', 'Sales', 'Customers', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'Year', 'Month', 'Day', 'Week', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd', 'CompetitionOpen', 'Promo2Active']


In [ ]:
print("----- Final feature check ----")
print(f"Shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nAll clean: {df.isnull().sum().sum() == 0}")
print(f"\nDtypes:\n{df.dtypes}")

----- Final feature check ----
Shape: (844338, 20)
Missing values:
Series([], dtype: int64)

All clean: True

Dtypes:
Store                    int64
DayOfWeek                int64
Sales                    int64
Customers                int64
Promo                    int64
StateHoliday             int64
SchoolHoliday            int64
StoreType                int64
Assortment               int64
CompetitionDistance    float64
Promo2                   int64
Year                     int32
Month                    int32
Day                      int32
Week                     int64
IsWeekend                int64
IsMonthStart             int64
IsMonthEnd               int64
CompetitionOpen        float64
Promo2Active             int64
dtype: object


In [ ]:
os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/train_features.csv", index=False)

print("Saved to ../data/processed/train_features.csv")
print(f"File size: {os.path.getsize('../data/processed/train_features.csv') / 1024 / 1024:.1f} MB")

Saved to ../data/processed/train_features.csv
File size: 49.8 MB


## Feature Engineering Summary

### Date features (7 new columns)
- Year, Month, Day, Week — extracted from Date
- IsWeekend — 1 if DayOfWeek is 6 or 7
- IsMonthStart, IsMonthEnd — flags for start and end of month

### Competition feature (1 new column)
- CompetitionOpen — months since nearest competitor opened
  Calculated as: 12 * (Year - CompetitionOpenSinceYear) + (Month - CompetitionOpenSinceMonth)
  Negative values clipped to 0 (stores with no competition data)

### Promo2 feature (1 new column)
- Promo2Active — 1 if store is actively running Promo2 on this date
  Checks: store participates in Promo2, current month is in PromoInterval,
  and current date is on or after Promo2SinceYear/Week

### Encoding
- StoreType: a=0, b=1, c=2, d=3
- Assortment: a=0, b=1, c=2
- StateHoliday: 0=0, a=1, b=2, c=3

### Columns dropped (7)
Date, Open, CompetitionOpenSinceMonth, CompetitionOpenSinceYear,
Promo2SinceWeek, Promo2SinceYear, PromoInterval

### Final output
- Shape: (844,338, 20)
- Missing values: 0
- Saved to: data/processed/train_features.csv